In [0]:
%python
dbutils.widgets.text("catalog","default")
catalog=dbutils.widgets.get("catalog")

In [0]:
%sql
create table if not exists fk_1368.bronze.stores
using delta
comment "raw stores csv files";

copy into fk_1368.bronze.stores
from (
    select *, _metadata.file_path as _source_file, current_timestamp() as _ingested_at
    from '/Volumes/fk_1368/landing/raw/stores/' )
fileformat = csv
format_options ('header' = 'true')
copy_options ('mergeSchema' = 'true')

In [0]:
%sql
create table if not exists fk_1368.bronze.customers
using delta
comment "raw customers csv files";

copy into fk_1368.bronze.customers
from (
    select *, _metadata.file_path as _source_file, current_timestamp() as _ingested_at
    from '/Volumes/fk_1368/landing/raw/customers/' )
fileformat = csv
format_options ('header' = 'true')
copy_options ('mergeSchema' = 'true')

In [0]:
%sql
create table if not exists fk_1368.bronze.products
using delta
comment "raw products json files";

COPY INTO fk_1368.bronze.products
  FROM '/Volumes/fk_1368/landing/raw/products/'
  FILEFORMAT = JSON
  FILES = ('products_2026-03-01.json')
  FORMAT_OPTIONS ('multiLine' = 'true')
  COPY_OPTIONS ('mergeSchema' = 'true')

In [0]:
# Create the target Bronze table
source_path = "/Volumes/fk_1368/landing/raw/orders/"
target_table = "fk_1368.bronze.orders"
checkpoint_path = "/Volumes/fk_1368/ops/pipeline_state/checkpoints/orders"
schema_path = "/Volumes/fk_1368/ops/pipeline_state/schema/orders"

In [0]:
# Read files 
df_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaLocation", schema_path)
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("cloudFiles.rescuedDataColumn", "_rescued_data")
    .load(source_path)
)

In [0]:
# Add metadata columns for file path and ingestion timestamp
from pyspark.sql.functions import col, current_timestamp

df_stream = (
    df_stream
    .withColumn("_source_file", col("_metadata.file_path"))
    .withColumn("_ingested_at", current_timestamp())
)

# Write to Bronze Delta table
query = (
    df_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(target_table)
)

In [0]:
%sql
SELECT COUNT(*) AS total_rows

FROM fk_1368.bronze.orders;